#### *Performance Metrics Calculation*
##### Implementation of PSNR, SSIM and Histogram plot to evaluate the quality of a stego-image against the original cover image for DCT.

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim

def calculate_psnr(cover, stego):
    # Calculate spatial pixel difference
    mse = np.mean((cover.astype(np.float64) - stego.astype(np.float64)) ** 2)
    if mse == 0:
        return float('inf')
    return 10 * np.log10((255.0 ** 2) / mse)

def calculate_ssim(cover, stego):
    if len(cover.shape) == 3:
        return ssim(cover, stego, channel_axis=-1)
    return ssim(cover, stego)

def extract_dct_coefficients(img_grayscale):
    """
    Divides the image into 8x8 blocks, computes the DCT, 
    and collects all AC coefficients.
    """
    h, w = img_grayscale.shape
    # Ensure dimensions are multiples of 8
    h, w = (h // 8) * 8, (w // 8) * 8
    img_cropped = img_grayscale[:h, :w].astype(np.float32)
    
    ac_coefficients = []
    
    # Process each 8x8 block
    for i in range(0, h, 8):
        for j in range(0, w, 8):
            block = img_cropped[i:i+8, j:j+8]
            dct_block = cv2.dct(block)
            
            # Quantize roughly if your pipeline uses a quantization table (e.g., JPEG standard)
            # For demonstration, we look at raw or rounded DCT coefficients
            quantized_block = np.round(dct_block) 
            
            # Extract AC coefficients (exclude top-left DC component at [0,0])
            ac_coeffs = np.delete(quantized_block.flatten(), 0)
            ac_coefficients.extend(ac_coeffs)
            
    return np.array(ac_coefficients)

def run_dct_analysis(cover_path, stego_path):
    # Load images in Grayscale for classic DCT analysis (or isolate the Y/Luma channel)
    cover = cv2.imread(cover_path, cv2.IMREAD_GRAYSCALE)
    stego = cv2.imread(stego_path, cv2.IMREAD_GRAYSCALE)
    
    if cover is None or stego is None:
        print("Error loading images.")
        return

    # 1. Compute Spatial Metrics
    psnr_val = calculate_psnr(cover, stego)
    ssim_val = calculate_ssim(cover, stego)
    print(f"--- Spatial Domain Analysis ---")
    print(f"PSNR: {psnr_val:.4f} dB")
    print(f"SSIM: {ssim_val:.4f}\n")
    
    # 2. Extract Frequency Coefficients
    cover_ac = extract_dct_coefficients(cover)
    stego_ac = extract_dct_coefficients(stego)
    
    # 3. Plot Histograms
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Spatial Histogram (Pixels)
    axes[0].hist(cover.ravel(), bins=256, range=[0,256], color='blue', alpha=0.5, label='Cover Pixels')
    axes[0].hist(stego.ravel(), bins=256, range=[0,256], color='orange', alpha=0.5, label='Stego Pixels')
    axes[0].set_title("Spatial Pixel Histogram")
    axes[0].set_xlabel("Pixel Intensity")
    axes[0].set_ylabel("Frequency")
    axes[0].legend()
    
    # Frequency Histogram (DCT AC Coefficients)
    # Focus on coefficients near zero (-5 to 5) where data hiding changes are most visible
    axes[1].hist(cover_ac, bins=11, range=[-5, 5], color='blue', alpha=0.5, rwidth=0.8, label='Cover DCT')
    axes[1].hist(stego_ac, bins=11, range=[-5, 5], color='orange', alpha=0.5, rwidth=0.6, label='Stego DCT')
    axes[1].set_title("DCT AC Coefficient Histogram (Zoomed near 0)")
    axes[1].set_xlabel("Coefficient Value")
    axes[1].set_ylabel("Frequency")
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()

# Run execution
# run_dct_analysis('cover.png', 'stego.png')